In [ ]:
"""
==============================================================================
Asphalt Mix Design Analysis Suite
==============================================================================
Modules:
    A – Aggregate Gradation Automator
    B – Volumetric Properties Calculator
    C – IDEAL-CT Cracking Index Analyzer
    D – Rutting Analyzer (Hamburg Wheel Tracking Test)

Primary packages : numpy, pandas, matplotlib, seaborn
Additional package: scipy  (numerical integration, signal smoothing,
                            linear regression)
==============================================================================
"""

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import integrate, signal
from scipy.stats import linregress

# ── Global style ─────────────────────────────────────────────────────────── #
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 130, "figure.figsize": (11, 6.5)})


# ============================================================================ #
#  MODULE A – Aggregate Gradation Automator                                    #
# ============================================================================ #

def run_module_a(filepath: str, blend_overrides: dict = None) -> pd.DataFrame:
    """
    Read individual aggregate sieve data and blend percentages, compute the
    combined gradation, and plot it against NDOT specification control points.

    Parameters
    ----------
    filepath : str
        Path to the Aggregate_Gradation Excel file.
    blend_overrides : dict, optional
        ``{aggregate_name_substring: fraction}`` to override the blend ratios
        in the file.  Fractions must sum to 1.0.  If None the file values are
        used as-is.

    Returns
    -------
    pd.DataFrame
        Combined gradation with Spec_Min, Spec_Max, and Status columns.
    """
    print("\n" + "=" * 70)
    print("  MODULE A — Aggregate Gradation Automator")
    print("=" * 70)

    # ── 1. Load raw spreadsheet (no header parsing) ───────────────────── #
    raw = pd.read_excel(filepath, sheet_name=0, header=None)
    # Row 0 : merged "Sieve Analysis (% Passing)" title — skip
    # Row 1 : actual column headers → sieve sizes in mm starting at col 2
    # Rows 2-7 : aggregate data rows
    # Row 8 : Spec Min
    # Row 9 : Spec Max

    header_row = raw.iloc[1]
    sieve_sizes_mm = np.array([
        float(header_row.iloc[c])
        for c in range(2, raw.shape[1])
        if pd.notna(header_row.iloc[c])
    ])

    # ── 2. Parse aggregate rows ───────────────────────────────────────── #
    agg_names, blend_pcts, agg_passing = [], [], []
    spec_min_vals, spec_max_vals = None, None
    spec_count = 0

    for _, row in raw.iloc[2:].iterrows():
        name_raw = str(row.iloc[0]).strip()
        pct_raw  = row.iloc[1]

        # Specification rows: name is NaN
        if name_raw in ("nan", ""):
            spec_vals = np.array([
                float(row.iloc[c]) if pd.notna(row.iloc[c]) else np.nan
                for c in range(2, 2 + len(sieve_sizes_mm))
            ])
            if spec_count == 0:
                spec_min_vals = spec_vals
            elif spec_count == 1:
                spec_max_vals = spec_vals
            spec_count += 1
            continue

        pct_str = str(pct_raw).strip()
        if pct_str.lower() in ("nan", "?", "sapecification range", ""):
            continue

        # Skip the combined-gradation row
        if "combined" in name_raw.lower():
            continue

        try:
            pct = float(pct_str)
        except ValueError:
            continue

        passing = np.array([
            float(row.iloc[c]) if pd.notna(row.iloc[c]) else np.nan
            for c in range(2, 2 + len(sieve_sizes_mm))
        ])

        agg_names.append(name_raw)
        blend_pcts.append(pct)
        agg_passing.append(passing)

    blend_pcts  = np.array(blend_pcts)
    agg_passing = np.array(agg_passing)

    # ── 3. Apply user blend overrides ────────────────────────────────── #
    if blend_overrides is not None:
        for name, frac in blend_overrides.items():
            idx = [i for i, n in enumerate(agg_names) if name.lower() in n.lower()]
            if idx:
                blend_pcts[idx[0]] = frac
        total = blend_pcts.sum()
        if not np.isclose(total, 1.0, atol=0.01):
            print(f"  Warning: Blend fractions sum to {total:.4f} — expected 1.0")

    # ── 4. Compute combined gradation ────────────────────────────────── #
    combined = np.nansum(agg_passing * blend_pcts[:, np.newaxis], axis=0)

    # ── 5. Build results table ───────────────────────────────────────── #
    result_df = pd.DataFrame({
        "Sieve_mm":        sieve_sizes_mm,
        "Combined_%Pass":  np.round(combined * 100, 2),
        "Spec_Min_%":      np.round(spec_min_vals * 100, 2) if spec_min_vals is not None else np.nan,
        "Spec_Max_%":      np.round(spec_max_vals * 100, 2) if spec_max_vals is not None else np.nan,
    })

    def _check(row):
        c  = row["Combined_%Pass"]
        lo = row["Spec_Min_%"]
        hi = row["Spec_Max_%"]
        if pd.isna(lo) or pd.isna(hi):
            return "—"
        return "PASS" if lo <= c <= hi else "FAIL"

    result_df["Status"] = result_df.apply(_check, axis=1)

    print("\n  Blend ratios:")
    for name, pct in zip(agg_names, blend_pcts):
        print(f"    {name:<28} {pct * 100:.1f}%")

    print("\n  Combined Gradation vs. NDOT Control Points:")
    print(result_df.to_string(index=False))

    # ── 6. Plot ──────────────────────────────────────────────────────── #
    fig, ax = plt.subplots(figsize=(12, 7))
    palette = sns.color_palette("muted", n_colors=len(agg_names))

    for i, (name, pct) in enumerate(zip(agg_names, blend_pcts)):
        ax.plot(sieve_sizes_mm, agg_passing[i] * 100,
                linestyle="--", linewidth=1.2, color=palette[i],
                label=f"{name} ({pct * 100:.0f}%)", alpha=0.7)

    ax.plot(sieve_sizes_mm, combined * 100,
            color="black", linewidth=2.5, marker="o", markersize=5,
            label="Combined Gradation", zorder=5)

    if spec_min_vals is not None and spec_max_vals is not None:
        valid = ~np.isnan(spec_min_vals) & ~np.isnan(spec_max_vals)
        if valid.any():
            ax.fill_between(sieve_sizes_mm[valid],
                            spec_min_vals[valid] * 100,
                            spec_max_vals[valid] * 100,
                            alpha=0.15, color="green", label="NDOT Spec Band")
            ax.plot(sieve_sizes_mm[valid], spec_min_vals[valid] * 100,
                    color="green", linewidth=1.5, linestyle=":", label="Spec Min")
            ax.plot(sieve_sizes_mm[valid], spec_max_vals[valid] * 100,
                    color="green", linewidth=1.5, linestyle="-.", label="Spec Max")

    ax.set_xscale("log")
    pos_sieve = sieve_sizes_mm[sieve_sizes_mm > 0]
    ax.set_xlim(sieve_sizes_mm.max() * 1.3, pos_sieve.min() * 0.4)
    ax.set_xlabel("Sieve Size (mm)  —  Log Scale", fontsize=12)
    ax.set_ylabel("% Passing", fontsize=12)
    ax.set_title("Module A — Aggregate Gradation Curve\nvs. NDOT Control Points", fontsize=13)
    ax.legend(loc="upper left", fontsize=8, ncol=2)
    ax.set_ylim(-1, 106)
    ax.grid(True, which="both", linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.savefig("Module_A_Gradation.png", bbox_inches="tight")
    plt.show()
    print("\n  Plot saved: Module_A_Gradation.png")

    return result_df


# ============================================================================ #
#  MODULE B – Volumetric Properties Calculator                                 #
# ============================================================================ #

def run_module_b(filepath: str) -> pd.DataFrame:
    """
    Read Gmm and Gmb measurements and compute Air Voids for each sample.

    Formulae (AASHTO T 209 / T 166):
        Gmm = A_rice / (A_rice - B_rice)   [Rice specific gravity]
        Gmb = A_dry  / (B_ssd - C_sub)     [Bulk specific gravity]
        Va  = 100 * (Gmm - Gmb) / Gmm      [Air voids, %]

    Notes: Gmm data is measured once per batch (Sample 1 only); Samples 2
    and 3 inherit the same Gmm value, which is standard practice.

    Quality Control: Va outside 6.5–7.5 % triggers an "Invalid" warning.

    Parameters
    ----------
    filepath : str  – Path to the Volumetrics Excel file.

    Returns
    -------
    pd.DataFrame  – Gmm, Gmb, Va (%), and Status per sample.
    """
    print("\n" + "=" * 70)
    print("  MODULE B — Volumetric Properties Calculator")
    print("=" * 70)

    raw = pd.read_excel(filepath, sheet_name=0, header=None)
    # Column layout (0-indexed):
    #  0  Sample ID
    #  1  Bowl mass in air (g)
    #  2  Bowl submerged (g)
    #  3  Bowl + Mix in air (g)
    #  4  Bowl + Mix submerged (g)
    #  6  Dry mass of specimen (g)
    #  7  Submerged mass (g)
    #  8  SSD mass (g)

    data_rows = raw.iloc[2:].reset_index(drop=True)
    gmm_shared = None
    records    = []

    for _, row in data_rows.iterrows():
        sample_raw = row.iloc[0]
        if pd.isna(sample_raw):
            continue
        try:
            sample_id = int(sample_raw)
        except (ValueError, TypeError):
            continue

        # Gmb inputs
        try:
            A_dry = float(row.iloc[6])
            C_sub = float(row.iloc[7])
            B_ssd = float(row.iloc[8])
        except (ValueError, TypeError):
            print(f"  Warning: Sample {sample_id} — missing Gmb data, skipping.")
            continue

        Gmb = A_dry / (B_ssd - C_sub)

        # Gmm inputs (present only for Sample 1)
        bowl_air_v     = row.iloc[1]
        bowl_sub_v     = row.iloc[2]
        bowl_mix_air_v = row.iloc[3]
        bowl_mix_sub_v = row.iloc[4]

        has_rice = all(
            pd.notna(v) and str(v).strip() not in ("", "?", "nan")
            for v in [bowl_air_v, bowl_sub_v, bowl_mix_air_v, bowl_mix_sub_v]
        )

        if has_rice:
            bowl_air     = float(bowl_air_v)
            bowl_sub     = float(bowl_sub_v)
            bowl_mix_air = float(bowl_mix_air_v)
            bowl_mix_sub = float(bowl_mix_sub_v)
            A_rice = bowl_mix_air - bowl_air   # dry mix mass
            B_rice = bowl_mix_sub - bowl_sub   # net submerged mix mass
            Gmm = A_rice / (A_rice - B_rice)
            gmm_shared = Gmm
        elif gmm_shared is not None:
            Gmm = gmm_shared
        else:
            print(f"  Warning: Sample {sample_id} — Gmm unavailable, skipping.")
            continue

        Va = 100.0 * (Gmm - Gmb) / Gmm

        if 6.5 <= Va <= 7.5:
            status = "Valid for Performance Testing"
        else:
            status = "INVALID — Air Voids outside 6.5–7.5%"

        records.append({
            "Sample":  sample_id,
            "Gmm":     round(Gmm, 4),
            "Gmb":     round(Gmb, 4),
            "Va (%)":  round(Va, 2),
            "Status":  status,
        })

    results = pd.DataFrame(records)

    print("\n  Calculated Volumetric Properties:")
    print(results.to_string(index=False))

    invalid = results[~results["Status"].str.startswith("Valid")]
    if not invalid.empty:
        print("\n  WARNING — Samples invalid for performance testing:")
        for _, r in invalid.iterrows():
            print(f"    Sample {r['Sample']}: Va = {r['Va (%)']}%  -> {r['Status']}")
    else:
        print("\n  All samples are valid for performance testing.")

    # ── Plot ─────────────────────────────────────────────────────────── #
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["steelblue" if s.startswith("Valid") else "tomato"
              for s in results["Status"]]
    bars = ax.bar(results["Sample"].astype(str), results["Va (%)"],
                  color=colors, edgecolor="black", linewidth=0.8, width=0.45)
    ax.axhspan(6.5, 7.5, alpha=0.12, color="green",
               label="Target Va  (6.5–7.5%)")
    ax.axhline(6.5, color="green", linestyle="--", linewidth=1.3)
    ax.axhline(7.5, color="green", linestyle="--", linewidth=1.3)

    for bar, val in zip(bars, results["Va (%)"]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.07,
                f"{val:.2f}%", ha="center", va="bottom",
                fontsize=11, fontweight="bold")

    ax.set_xlabel("Sample ID", fontsize=12)
    ax.set_ylabel("Air Voids  Va (%)", fontsize=12)
    ax.set_title("Module B — Volumetric Properties\nAir Voids per Sample", fontsize=13)
    ax.legend(fontsize=10)
    ax.set_ylim(0, max(results["Va (%)"].max() + 1.8, 9.5))
    plt.tight_layout()
    plt.savefig("Module_B_Volumetrics.png", bbox_inches="tight")
    plt.show()
    print("\n  Plot saved: Module_B_Volumetrics.png")

    return results


# ============================================================================ #
#  MODULE C – IDEAL-CT Cracking Index Analyzer                                 #
# ============================================================================ #

def _compute_ct_index(load: np.ndarray, disp: np.ndarray,
                      thickness_mm: float, diameter_mm: float) -> dict:
    """
    Compute CT_index from one IDEAL-CT Load-Displacement curve.

    Algorithm
    ---------
    1. Work of fracture  Wf = area under Load-Displacement curve (scipy trapezoid)
    2. Gf = Wf / (t * D * 1e-6)   [J/m2]
    3. Peak load identified; post-peak 75% point found by linear interpolation.
    4. m75 = post-peak slope at 75% of peak (local linear regression, scipy).
    5. l75 = displacement from peak to 75% point  [mm].
    6. CT_index = (Gf / (|m75| * l75)) * (t / D^2)

    Parameters
    ----------
    load         : np.ndarray  [kN]
    disp         : np.ndarray  [mm]
    thickness_mm : float
    diameter_mm  : float

    Returns
    -------
    dict with all intermediate and final values.
    """
    # 1. Work of fracture and fracture energy
    Wf = integrate.trapezoid(load, disp)             # kN*mm = J
    Gf = Wf / (thickness_mm * diameter_mm * 1e-6)   # J/m2

    # 2. Peak
    peak_idx  = int(np.argmax(load))
    peak_load = load[peak_idx]
    target_75 = 0.75 * peak_load

    # 3. Post-peak 75% crossing
    post_disp = disp[peak_idx:]
    post_load = load[peak_idx:]
    disp_75   = post_disp[-1]
    idx_75    = len(post_load) - 1

    for i in range(1, len(post_load)):
        if post_load[i] <= target_75:
            d0, d1 = post_disp[i - 1], post_disp[i]
            l0, l1 = post_load[i - 1], post_load[i]
            frac    = (target_75 - l0) / (l1 - l0) if l1 != l0 else 0.0
            disp_75 = d0 + frac * (d1 - d0)
            idx_75  = i
            break

    l75 = disp_75 - disp[peak_idx]   # [mm]

    # 4. Local slope at 75% post-peak (scipy linregress)
    window = max(5, len(post_load) // 15)
    i_lo   = max(0,              idx_75 - window)
    i_hi   = min(len(post_load), idx_75 + window + 1)
    slope, *_ = linregress(post_disp[i_lo:i_hi], post_load[i_lo:i_hi])
    m75 = slope   # kN/mm (negative post-peak)

    # 5. CT_index
    CT_index = (Gf / (abs(m75) * l75)) * (thickness_mm / diameter_mm ** 2)

    return {
        "peak_load_kN":    round(float(peak_load), 4),
        "disp_at_peak_mm": round(float(disp[peak_idx]), 4),
        "Wf_J":            round(float(Wf), 4),
        "Gf_J_m2":         round(float(Gf), 2),
        "m75_kN_mm":       round(float(m75), 6),
        "l75_mm":          round(float(l75), 4),
        "disp_75_mm":      round(float(disp_75), 4),
        "load_75_kN":      round(float(target_75), 4),
        "CT_index":        round(float(CT_index), 4),
    }


def run_module_c(filepath: str) -> pd.DataFrame:
    """
    Parse the wide-format IDEAL-CT spreadsheet, compute CT_index per sample,
    and produce a Load vs. Displacement plot for all three samples.

    Parameters
    ----------
    filepath : str  – Path to the IDEAL-CT Excel file.

    Returns
    -------
    pd.DataFrame  – CT_index results for all samples.
    """
    print("\n" + "=" * 70)
    print("  MODULE C — IDEAL-CT Cracking Index Analyzer")
    print("=" * 70)

    raw = pd.read_excel(filepath, sheet_name=0, header=None)
    # Row 0: sample labels  (e.g. col 1 = "Sample 1 ( t = 61.91 mm, D = 150mm)")
    # Row 1: sub-headers    ("Load", "LLD_Avg" per sample pair)
    # Rows 2+: numeric data

    sample_header_row = raw.iloc[0].tolist()
    sample_groups = []
    for col_idx, cell in enumerate(sample_header_row):
        if pd.notna(cell) and "Sample" in str(cell):
            t_m = re.search(r"t\s*=\s*([\d.]+)", str(cell))
            d_m = re.search(r"D\s*=\s*([\d.]+)", str(cell))
            sample_groups.append({
                "name":      f"Sample {len(sample_groups) + 1}",
                "col_load":  col_idx,
                "col_disp":  col_idx + 1,
                "thickness": float(t_m.group(1)) if t_m else 61.91,
                "diameter":  float(d_m.group(1)) if d_m else 150.0,
            })

    data_rows = raw.iloc[2:].reset_index(drop=True)
    results_list = []

    fig, ax = plt.subplots(figsize=(12, 7))
    colors_ct = sns.color_palette("deep", n_colors=len(sample_groups))

    for k, sg in enumerate(sample_groups):
        raw_load = pd.to_numeric(data_rows.iloc[:, sg["col_load"]], errors="coerce")
        raw_disp = pd.to_numeric(data_rows.iloc[:, sg["col_disp"]], errors="coerce")
        mask     = raw_load.notna() & raw_disp.notna()
        load     = raw_load[mask].to_numpy(dtype=float)
        disp     = raw_disp[mask].to_numpy(dtype=float)

        if len(load) < 10:
            print(f"  Warning: {sg['name']} has insufficient data — skipping.")
            continue

        metrics = _compute_ct_index(load, disp, sg["thickness"], sg["diameter"])

        print(f"\n  {sg['name']}  "
              f"(t = {sg['thickness']} mm,  D = {sg['diameter']} mm)")
        for key, val in metrics.items():
            print(f"    {key:<22} = {val}")

        results_list.append({"Sample": sg["name"],
                              "Thickness_mm": sg["thickness"],
                              "Diameter_mm":  sg["diameter"],
                              **metrics})

        ax.plot(disp, load, color=colors_ct[k], linewidth=1.8, label=sg["name"])

        pk = int(np.argmax(load))
        ax.scatter(disp[pk], load[pk], color=colors_ct[k],
                   s=90, zorder=6, marker="^")
        ax.scatter(metrics["disp_75_mm"], metrics["load_75_kN"],
                   color=colors_ct[k], s=70, zorder=6, marker="v",
                   edgecolors="black", linewidths=0.7)

    ax.set_xlabel("Displacement / LLD Average (mm)", fontsize=12)
    ax.set_ylabel("Load (kN)", fontsize=12)
    ax.set_title("Module C — IDEAL-CT  Load vs. Displacement\n"
                 "Triangle-up = Peak Load    Triangle-down = 75% Post-Peak", fontsize=13)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig("Module_C_IDEAL_CT.png", bbox_inches="tight")
    plt.show()
    print("\n  Plot saved: Module_C_IDEAL_CT.png")

    results_df = pd.DataFrame(results_list)
    print("\n  CT_index Summary:")
    print(results_df[["Sample", "Gf_J_m2", "m75_kN_mm",
                       "l75_mm", "CT_index"]].to_string(index=False))
    return results_df


# ============================================================================ #
#  MODULE D – Rutting Analyzer (Hamburg Wheel Tracking Test)                   #
# ============================================================================ #

def _detect_sip(cycles: np.ndarray, rut: np.ndarray,
                sg_window: int = 51) -> tuple:
    """
    Detect the Stripping Inflection Point (SIP) in a HWTT rut-depth curve.

    Method
    ------
    The rut curve is smoothed with a Savitzky-Golay filter (scipy.signal).
    The first derivative (rutting rate) is computed via np.gradient.
    The SIP is the cycle of maximum first-derivative after the initial
    densification/creep phase (first 20% of data is skipped).

    Parameters
    ----------
    cycles    : np.ndarray  – Wheel cycle counts.
    rut       : np.ndarray  – Rut depth (mm).
    sg_window : int         – Savitzky-Golay window (odd, <= len(rut)).

    Returns
    -------
    (sip_cycle, sip_rut) as floats, or (None, None) if not detected.
    """
    sg_window = min(sg_window, len(rut) - (1 - len(rut) % 2))
    if sg_window % 2 == 0:
        sg_window -= 1
    sg_window = max(sg_window, 5)

    rut_smooth = signal.savgol_filter(rut, window_length=sg_window, polyorder=3)
    d1         = np.gradient(rut_smooth, cycles)

    skip = max(1, int(len(cycles) * 0.20))
    d1_search     = d1[skip:]
    cycles_search = cycles[skip:]
    rut_search    = rut[skip:]

    if d1_search.size == 0:
        return None, None

    sip_rel = int(np.argmax(d1_search))
    return float(cycles_search[sip_rel]), float(rut_search[sip_rel])


def run_module_d(filepath: str,
                 failure_threshold_mm: float = 12.5,
                 pass_cycles: int = 20_000) -> pd.DataFrame:
    """
    Analyze Hamburg Wheel Tracking Test data for each sample pair.

    Logic
    -----
    - SIP Detection: Savitzky-Golay smoothing + gradient maximum (scipy).
    - Failure Cycle: linear interpolation to exact cycle at 12.5 mm rut.
    - Pass/Fail: sample must survive 20,000 cycles below 12.5 mm.

    Parameters
    ----------
    filepath             : str   – Path to the HWTT Excel file.
    failure_threshold_mm : float – Rut depth failure threshold (default 12.5 mm).
    pass_cycles          : int   – Cycles required for Pass (default 20,000).

    Returns
    -------
    pd.DataFrame  – SIP, failure cycle, and Pass/Fail per sample.
    """
    print("\n" + "=" * 70)
    print("  MODULE D — Rutting Analyzer (Hamburg Wheel Tracking Test)")
    print("=" * 70)

    raw = pd.read_excel(filepath, sheet_name=0, header=None)
    # Row 0: pair group headers  (col 1 = "Pair 1", col 4 = "Pair 2", ...)
    # Row 1: column headers      ("LC", "RD")
    # Rows 2+: data

    group_row = raw.iloc[0].tolist()
    data_rows = raw.iloc[2:].reset_index(drop=True)

    pairs = []
    for col_idx, cell in enumerate(group_row):
        if pd.notna(cell) and str(cell).strip().lower().startswith("pair"):
            pairs.append({
                "name":   str(cell).strip(),
                "col_lc": col_idx,
                "col_rd": col_idx + 1,
            })

    records    = []
    palette_d  = sns.color_palette("Set2", n_colors=max(len(pairs), 2))
    fig, ax    = plt.subplots(figsize=(12, 7))

    for k, pair in enumerate(pairs):
        lc_raw = pd.to_numeric(data_rows.iloc[:, pair["col_lc"]], errors="coerce")
        rd_raw = pd.to_numeric(data_rows.iloc[:, pair["col_rd"]], errors="coerce")
        mask   = lc_raw.notna() & rd_raw.notna()
        cycles = lc_raw[mask].to_numpy(dtype=float)
        rut    = rd_raw[mask].to_numpy(dtype=float)

        if len(cycles) < 10:
            print(f"  Warning: {pair['name']} — insufficient data, skipping.")
            continue

        # SIP detection
        sip_cycle, sip_rut = _detect_sip(cycles, rut)

        # Failure cycle (first crossing of threshold) — linear interpolation
        fail_cycle = fail_rut = None
        above = np.where(rut >= failure_threshold_mm)[0]
        if above.size > 0:
            fi = above[0]
            if fi > 0:
                c0, c1 = cycles[fi - 1], cycles[fi]
                r0, r1 = rut[fi - 1],    rut[fi]
                frac       = (failure_threshold_mm - r0) / (r1 - r0)
                fail_cycle = c0 + frac * (c1 - c0)
                fail_rut   = failure_threshold_mm
            else:
                fail_cycle, fail_rut = float(cycles[fi]), float(rut[fi])

        # Pass / Fail verdict
        max_cycles = cycles.max()
        max_rut    = rut.max()
        if fail_cycle is not None and fail_cycle <= pass_cycles:
            verdict = f"FAIL  (threshold crossed at {fail_cycle:,.0f} cycles)"
        elif max_cycles >= pass_cycles and max_rut < failure_threshold_mm:
            verdict = "PASS"
        elif max_cycles < pass_cycles:
            verdict = f"INCONCLUSIVE (test ended at {max_cycles:,.0f} cycles)"
        else:
            verdict = "PASS"

        print(f"\n  {pair['name']}")
        print(f"    Cycles in dataset      : {max_cycles:,.0f}")
        print(f"    Maximum rut depth      : {max_rut:.3f} mm")
        if sip_cycle is not None:
            print(f"    SIP detected at        : {sip_cycle:,.0f} cycles "
                  f"| rut = {sip_rut:.3f} mm")
        else:
            print("    SIP                    : Not detected")
        if fail_cycle is not None:
            print(f"    Failure ({failure_threshold_mm} mm) at    : "
                  f"{fail_cycle:,.0f} cycles")
        else:
            print(f"    Failure ({failure_threshold_mm} mm)        : Not reached")
        print(f"    Verdict                : {verdict}")

        records.append({
            "Sample":     pair["name"],
            "Max_Cycles": int(max_cycles),
            "Max_Rut_mm": round(max_rut, 3),
            "SIP_Cycle":  round(sip_cycle) if sip_cycle else "N/A",
            "SIP_Rut_mm": round(sip_rut, 3) if sip_rut else "N/A",
            "Fail_Cycle": f"{fail_cycle:,.0f}" if fail_cycle else "N/R",
            "Verdict":    verdict,
        })

        ax.plot(cycles, rut, color=palette_d[k], linewidth=2,
                label=pair["name"])

        if sip_cycle is not None:
            ax.scatter(sip_cycle, sip_rut, color=palette_d[k],
                       s=100, zorder=7, marker="D",
                       edgecolors="black", linewidths=0.7)
            ax.annotate(
                f"SIP\n{sip_cycle:,.0f} cyc",
                xy=(sip_cycle, sip_rut),
                xytext=(sip_cycle + max_cycles * 0.05, sip_rut - 0.6),
                arrowprops=dict(arrowstyle="->", color=palette_d[k]),
                fontsize=8, color=palette_d[k], fontweight="bold",
            )

        if fail_cycle is not None:
            ax.scatter(fail_cycle, fail_rut, color=palette_d[k],
                       s=110, zorder=7, marker="X",
                       edgecolors="black", linewidths=0.7)
            ax.annotate(
                f"Fail\n{fail_cycle:,.0f} cyc",
                xy=(fail_cycle, fail_rut),
                xytext=(fail_cycle + max_cycles * 0.03, fail_rut + 0.5),
                arrowprops=dict(arrowstyle="->", color=palette_d[k]),
                fontsize=8, color=palette_d[k], fontweight="bold",
            )

    ax.axhline(failure_threshold_mm, color="red", linestyle="--",
               linewidth=1.5, label=f"Failure Threshold ({failure_threshold_mm} mm)")
    ax.axvline(pass_cycles, color="gray", linestyle=":",
               linewidth=1.3, label=f"Pass Limit ({pass_cycles:,} cycles)")

    ax.set_xlabel("Wheel Passes (Cycles)", fontsize=12)
    ax.set_ylabel("Rut Depth (mm)", fontsize=12)
    ax.set_title("Module D — Hamburg Wheel Tracking Test\n"
                 "Rut Depth vs. Wheel Cycles   Diamond = SIP   X = Failure Point",
                 fontsize=13)
    ax.legend(fontsize=9)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig("Module_D_HWTT.png", bbox_inches="tight")
    plt.show()
    print("\n  Plot saved: Module_D_HWTT.png")

    results_df = pd.DataFrame(records)
    print("\n  HWTT Summary:")
    print(results_df.to_string(index=False))
    return results_df


# ============================================================================ #
#  MAIN                                                                         #
# ============================================================================ #

if __name__ == "__main__":

    FILE_AGG   = "Aggregate_Gradation_csv.xlsx"
    FILE_VOL   = "Volumetrics_csv.xlsx"
    FILE_IDEAL = "IDEAL-CT_csv.xlsx"
    FILE_HWTT  = "HWTT_csv.xlsx"

    df_a = run_module_a(FILE_AGG)
    df_b = run_module_b(FILE_VOL)
    df_c = run_module_c(FILE_IDEAL)
    df_d = run_module_d(FILE_HWTT)

    print("\n" + "=" * 70)
    print("  All four modules complete.")
    print("  Output plots: Module_A_Gradation.png")
    print("                Module_B_Volumetrics.png")
    print("                Module_C_IDEAL_CT.png")
    print("                Module_D_HWTT.png")
    print("=" * 70)